In [2]:
from sklearn.linear_model import LinearRegression
from bokeh.models import HoverTool 
from bokeh.io import export_svg
import numpy as np
import transforms3d as td
import pandas as pd
import bokeh.io
import bokeh.plotting
# Import libraries
from mpl_toolkits import mplot3d
import matplotlib.pyplot as plt
import re
import panel as pn
from bokeh.models import ColumnDataSource
from bokeh.palettes import Viridis
pn.extension()
import plotly.graph_objects as go
from bokeh.io import export_png
import holoviews as hv
from bokeh.models import CategoricalColorMapper, Legend
from bokeh.layouts import column, row
from bokeh.models import LinearAxis, Range1d
from parse_patient_data import PatientData
bokeh.io.output_notebook()

Loading BokehJS ...

In [4]:
p1_opens = {'passive_start' : [[90, 112], [211, 236], [318, 348], [438, 468], [547, 580], [671, 702], [784, 811], [888, 919], [1000, 1030], [1105, 1136], [1217, 1252]],
            'active_rorcr' : [[1869, 1908], [2524, 2566], [3273, 3308]],
            'active_cube' : [[633, 668], [1025, 1057], [1971, 2000], [2092, 2143], [2291, 2319], [3050, 3079], [3207, 3241], [3357, 3382], [4863, 4895], [5581, 5913], [6008, 6035], [6204, 6234], [6578, 6604], [6881, 6913], [7294, 7320], [7589, 7611], [8336, 8365], [9151, 9179], [9360, 9374], [9825, 9854], [10394, 10413], [10736, 10766], [11095, 11124], [11916, 11931], [16373, 16405], [17230, 17270], [17804, 17830], [18271, 18303], [18730, 18755], [24525, 24576], [26133, 26181]],
            'passive_end' : [[226, 265], [324, 350], [420, 460], [510, 542], [598, 627], [711, 747], [815, 841], [898, 937], [1004, 1045], [1105, 1135]]}

p13_opens = {'passive_start' : [[350, 385], [572, 600], [752, 786], [906, 934], [1045, 1075], [1180, 1209], [1312, 1345], [1448, 1478], [1571, 1599], [1695, 1728]],
             'active_rorcr' : [[2924, 2943], [4940, 4955], [5354, 5377], [6881, 6895]],
             'active_cube' : [[3483, 3499], [5476, 5491], [6269, 6283], [7012, 7041], [7473, 7498], [8209, 8227], [8489, 8502], [8679, 8692], [9599, 9620], [10555, 10569], [10628, 10647], [12731, 12747]],
             'passive_end' : [[144, 167], [337, 358], [505, 521], [686, 702], [862, 883], [1035, 1052], [1206, 1223], [1353, 1374], [1523, 1538], [1668, 1689]]}

p3_opens = {'passive_start': [[150, 196], [384, 432], [568, 606], [751, 794], [928, 963], [1097, 1142], [1271, 1311], [1442, 1483], [1592, 1631], [1751, 1804]],
            'active_rorcr': [[1868, 1922], [2724, 2753], [3503, 3532]],
            'active_cube' : [[2588, 2635], [5546, 5601], [7733, 7788], [17792, 17809], [18952, 18966], [20984, 21007]],
            'passive_end' : [[87, 142], [263, 324], [469, 528], [662, 703], [854, 899], [1050, 1091], [1232, 1281], [1419, 1454], [1590, 1633], [1766, 1807], [1942, 1982]]}

opens_dict = {'p1' : p1_opens, 'p13': p13_opens, 'p3' : p3_opens}


import pickle

try:
    for p, p_dict in zip(['1', '13', '3'], [p1_opens, p13_opens, p3_opens]):
        pickle_file = open('p%s_opens'%p, 'wb')
        pickle.dump(p_dict, pickle_file)
        pickle_file.close()

except:
    print("Unable to write to file")

In [7]:
filename_1 = '/Users/kate/Downloads/p1_compiled_data.csv'
filename_3 = '/Users/kate/Downloads/p3_compiled_data.csv'
filename_13 = '/Users/kate/Downloads/p13_compiled_data.csv'

p1_data = PatientData(filename_1, p1_opens)
p3_data = PatientData(filename_3, p3_opens)
p13_data = PatientData(filename_13, p13_opens)
#print("Conditions: %s\nCalibration conditions: %s"%(p1_data.conditions, p1_data.calibration_conditions))

Condition: passive_start

Variance ratio: [0.85248566 0.14107915 0.00643519]
----------------------------

Condition: active_rorcr

Variance ratio: [0.89906019 0.07219195 0.02874787]
----------------------------

Condition: active_cube

Variance ratio: [0.89906019 0.07219195 0.02874787]
----------------------------

Condition: passive_end

Variance ratio: [0.70010885 0.28276654 0.0171246 ]
----------------------------

Condition: passive_start

Variance ratio: [0.88322349 0.1089908  0.00778571]
----------------------------

Condition: active_rorcr

Variance ratio: [0.92020428 0.05115604 0.02863968]
----------------------------

Condition: active_cube

Variance ratio: [0.92020428 0.05115604 0.02863968]
----------------------------

Condition: passive_end

Variance ratio: [0.85949466 0.11135786 0.02914748]
----------------------------

Condition: passive_start

Variance ratio: [0.90776818 0.08042244 0.01180938]
----------------------------

Condition: active_rorcr

Variance ratio: [0.855

In [169]:
class PlotData(object):
    def __init__(self, patient):
        self.data = patient
        self.plot_label = {'passive_start': 'Passive 1',
                            'passive_end': 'Passive 2',
                            'active_rorcr': 'Active 1',
                            'active_rorcr1': 'T4',
                            'active_cube': 'Active 2',
                            'mcp_angle': 'MCP Angle (deg)',
                            'pip_angle': 'PIP Angle (deg)',
                            'stiffness': 'Index Finger Stiffness (N/mm)',
                            'mcp_m' : 'MCP Joint Stiffness (Nmm/deg)',
                            'pip_m' : 'PIP Joint Stiffness (Nmm/deg)',
                            'torque_mcp' : 'MCP Joint Torque (Nmm)',
                            'torque_pip' : 'PIP Joint Torque (Nmm)',
                            'displacement': 'Cable Displacement (mm)',
                            'motor_position' : 'Cable Retraction (mm)',
                            'futek': 'Force (N)',
                            'net_force' : 'Net Force (N)',
                            'time_open': 'Time over Experimental Condition (unscaled)',
                            'time_unitless' : 'Time over Experiment',
                            'p1': 'S1',
                            'p13': 'S2',
                            'p3': 'S3',}

    def p_format(self, p):
        p.title.text_font_size = '30pt'
        p.title.align = 'center'
        p.legend.label_text_font_size = '25pt'
        p.xaxis.axis_label_text_font_size = '25pt'
        p.yaxis.axis_label_text_font_size = '25pt'
        p.yaxis.major_label_text_font_size = '20pt'
        p.xaxis.major_label_text_font_size = '20pt'
        return p

    def plot_stiffness(self, title = 'Index Finger Stiffness'):

        hover = HoverTool(tooltips=[("index", "$index"),])
        p = bokeh.plotting.figure(
                width=1000,
                height=600,
                x_axis_label= 'Opening Cycle #',
                y_axis_label=self.plot_label['stiffness'],
                title = title,
                tools = [hover, 'pan', 'wheel_zoom', 'box_zoom', 'reset'],
                y_range = [0, 0.87]
            )
        p.add_layout(Legend(), 'right')
        color = bokeh.palettes.d3['Category10'][10]
        n = 0

        for i, condition in enumerate(self.data.conditions):
            p.circle(np.arange(n, len(self.data.stiffnesses[condition])+n), self.data.stiffnesses[condition], legend_label = self.plot_label[condition], color = color[i], alpha = 1, size =12)
            #p.line(np.arange(n, len(self.data.stiffnesses[condition])+n), self.data.stiffnesses[condition], legend_label = condition, color = color[i], alpha = 0.5)
            p.segment(x0=np.arange(n, len(self.data.stiffnesses[condition])+n),x1=np.arange(n, len(self.data.stiffnesses[condition])+n), y0=0, y1 = self.data.stiffnesses[condition], 
                      legend_label = self.plot_label[condition], 
                      color = color[i], 
                      alpha = 0.5, 
                      line_width = 3)
            n += len(self.data.stiffnesses[condition])
        
        p = self.p_format(p)

        return p

    def plot_angle_angle(self, save = False, legend = True):
        '''
        Plots angle-angle plot with MCP angle on the x and PIP angle on the y
        '''

        hover = HoverTool(tooltips=[("index", "$index"),])
        p = bokeh.plotting.figure(
                width=600,
                height=600,
                x_axis_label="MCP Angle",
                y_axis_label="PIP Angle",
                title = ' MCP and PIP angles',
                x_range = [-45, 110],
                y_range = [-45, 110],
                tools = [hover, 'pan', 'wheel_zoom', 'box_zoom', 'reset']
            )
        color = bokeh.palettes.d3['Category10'][10]

        for i, condition in enumerate(self.data.conditions):
            for df in self.data.opens_dfs[condition][:3]:
                p.line(
                    df['mcp_angle'],
                    df['pip_angle'],
                    color=color[i],
                    legend_label=condition,
                    line_width=1,
                )
                p.circle(
                    df['mcp_angle'],
                    df['pip_angle'],
                    color=color[i],
                    legend_label=condition,
                    line_width=2,
                )
        p.vbar(x= np.mean([-45, 110]), top = -45, alpha = 0.07, color = 'grey', width = 45+110)
        p.vbar(x=np.mean([-45, 0]), top=110, alpha=0.07, color="grey", width=45)
        p.vbar(x=np.mean([-45, 0]), top=-45, alpha=0.1, color="grey", width=45)
        p.legend.click_policy='hide'
        #p = self.p_format(p)

        return p

    def plot_best_fit(self, save = False, legend = True):
        x = 'displacement'
        y = 'net_force'

        return p
    
    def plot_emg(self, df):
        p = bokeh.plotting.figure(
                width=800,
                height=600,
                y_axis_label='EMG',
                x_axis_label=self.plot_label[x],
                title = '%s EMG vs. Displacement'%self.plot_label[self.patient],
                tools = [hover, 'pan', 'wheel_zoom', 'box_zoom', 'tap', 'reset']
            )

        return p

    def plot_opens(self, open_set = False, save = False, x_max = 25, y_max = 12, n_start = 0, n_end = 10):
        hover = HoverTool(tooltips=[("index", "$index"),])
        p = bokeh.plotting.figure(
                width=800,
                height=600,
                x_axis_label="Cable Displacement (mm)",
                y_axis_label="Force (N)",
                title = 'Force vs. Displacement',
                x_range = [0, x_max],
                y_range = [0, y_max],
                tools = [hover, 'pan', 'wheel_zoom', 'box_zoom', 'reset']
            )
        color = bokeh.palettes.d3['Category10'][10]
        # Keep legend outside of plot
        p.add_layout(Legend(), 'right')
        
        for i, condition in enumerate(self.data.conditions):
            for df in self.data.opens_dfs[condition][n_start:n_end]:
                p.line(
                    df['displacement'],
                    df['futek'],
                    color=color[i],
                    legend_label=self.plot_label[condition],
                    line_width=3,
                    alpha=0.5
                )
                p.circle(
                    df['displacement'],
                    df['futek'],
                    color=color[i],
                    legend_label=self.plot_label[condition],
                    line_width=5,
                    alpha=0.3
                )
            
                
        #p.vbar(x= np.mean([-45, 110]), top = -45, alpha = 0.07, color = 'grey', width = 45+110)
        #p.vbar(x=np.mean([-45, 0]), top=110, alpha=0.07, color="grey", width=45)
        #p.vbar(x=np.mean([-45, 0]), top=-45, alpha=0.1, color="grey", width=45)
        p.legend.click_policy='hide'
        p = self.p_format(p)

        return p
    
    def plot_fit(self, open_set = False, save = False, x_max = 25, y_max = 12, n_start = 0, n_end = 10):
        hover = HoverTool(tooltips=[("index", "$index"),])
        p = bokeh.plotting.figure(
                width=800,
                height=600,
                x_axis_label="Cable Displacement (mm)",
                y_axis_label="Force (N)",
                title = 'Force vs. Displacement',
                x_range = [0, x_max],
                y_range = [0, y_max],
                tools = [hover, 'pan', 'wheel_zoom', 'box_zoom', 'reset']
            )
        color = bokeh.palettes.d3['Category10'][10]
        # Keep legend outside of plot
        p.add_layout(Legend(), 'right')
        
        for i, condition in enumerate(self.data.conditions):
            for df in self.data.opens_dfs[condition]:
                p.line(
                    df['displacement'],
                    df['futek'],
                    color=color[i],
                    legend_label=self.plot_label[condition],
                    line_width=1.5,
                    alpha=0.3
                )
                p.circle(
                    df['displacement'],
                    df['futek'],
                    color=color[i],
                    legend_label=self.plot_label[condition],
                    size=0,
                    alpha=0.3
                )
            m, b = self.data.avg_fit[condition]
            x = np.arange(0, x_max)
            p.line(
                x,
                m*x + b,
                color=color[i],
                legend_label=self.plot_label[condition],
                line_width=5,
                alpha=1)
            
                
        #p.vbar(x= np.mean([-45, 110]), top = -45, alpha = 0.07, color = 'grey', width = 45+110)
        #p.vbar(x=np.mean([-45, 0]), top=110, alpha=0.07, color="grey", width=45)
        #p.vbar(x=np.mean([-45, 0]), top=-45, alpha=0.1, color="grey", width=45)
        p.legend.click_policy='hide'
        p = self.p_format(p)

        return p

p1 = PlotData(p1_data)
p3 = PlotData(p3_data)
p13 = PlotData(p13_data)

In [134]:
ps = []
for i, p in enumerate([p1, p13, p3]):
    p_plot = p.plot_stiffness()
    p_plot.width = 750
    p_plot.height = 600
    p_plot.title = "Subject %i"%(i+1)
    p_plot.title.text_font_size = '30pt'
    p_plot.title.align = 'center'
    p_plot.toolbar_location = None
    p_plot.legend.visible = False
    ps.append(p_plot)

In [136]:
# Export Best fit
export_png(row(ps), filename = 'forcedisplacement_row.png')

'/Users/kate/Downloads/forcedisplacement_row.png'

In [187]:
p = p1.plot_stiffness()
p.width = 830
p.height = 600
p.legend.visible = False
p.toolbar_location = None
bokeh.io.show(p)

In [171]:
p = p13.plot_fit(n_start = 1, n_end = 2, x_max = 25, y_max = 7)
p.width = 800
p.height = 600
p.title = 'Subject 2'
p.title.text_font_size = '30pt'
p.title.align = 'center'
p.legend.visible = False
p.toolbar_location = None
bokeh.io.show(p)

In [159]:
p = p13.plot_opens(n_start = 1, n_end = 2, x_max = 25, y_max = 7)
p.width = 800
p.height = 600
p.title = 'Subject 2'
p.title.text_font_size = '30pt'
p.title.align = 'center'
p.legend.visible = False
p.toolbar_location = None
bokeh.io.show(p)

In [189]:
export_png(p, filename = 'stiffness.png')

'/Users/kate/Downloads/stiffness.png'